# 05 — BERTopic for wondr

Topic modeling pipeline untuk wondr by BNI menggunakan BERTopic dengan IndoBERT embeddings.

**Pipeline overview:**
1. Load preprocessed data + generate (or load cached) IndoBERT embeddings
2. Sensitivity analysis 3×3 grid (UMAP n_neighbors × HDBSCAN min_cluster_size)
3. Pick winner via c_v coherence (filter by topic count 15-30)
4. Validate winner with c_npmi
5. Final BERTopic fit with winner params
6. Inspect topics, refine domain stopwords (Phase B)
7. Re-fit with extended stopwords
8. DTM (topics_over_time) — monthly granularity
9. Cosine similarity matrices per topic (semantic drift)

**Methodology reference:** `bertopic_session_handoff_v2.md`

## Section 0 — Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

from utils.embedding import load_indobert_model, generate_embeddings

## Section 1 — Load data + Generate embeddings

**Heads up:** First run akan generate embeddings dari scratch (~20-30 menit untuk wondr di CPU). Subsequent runs akan load dari cache (`data/embeddings/wondr_embeddings.npy`) — instant.

### 1.1 Load preprocessed data

In [4]:
df_wondr = pd.read_csv('data/processed/wondr_bertopic.csv')

print(f"Shape: {df_wondr.shape}")
print(f"Columns: {df_wondr.columns.tolist()}")
df_wondr.head()

Shape: (8982, 6)
Columns: ['review_id', 'review_text_cleaned', 'relative_month', 'relative_week', 'date_wib', 'rating']


,review_id,review_text_cleaned,relative_month,relative_week,date_wib,rating
0,05c4afd9-2b73-462f-93ed-ad2f853647b5,kok tidak bisa di screenshot bagaimana caranya...,1,1,2024-07-05 16:24:39,2
1,e4786af6-59c9-4f34-897f-d20f8c5f412b,mau daftar susah disuruh telpon bni tapi tidak...,1,1,2024-07-05 16:28:04,1
2,cc78e674-330f-46f1-b279-27a9cbe2d823,tidak bisa buka rekening padahal no hp sudah s...,1,1,2024-07-05 21:14:52,1
3,2441c0dc-eda3-4f2c-8abe-f860c951af50,fitur paling basic untuk cetak statement bulan...,1,1,2024-07-05 22:04:41,1
4,0afc898b-3b64-426d-acaf-ae6b3c4bc967,sangat bagus sekali tapi tidak ada pitur setor...,1,1,2024-07-05 23:54:54,2


In [5]:
# Sanity check: no nulls in critical columns
critical_cols = ['review_text_cleaned', 'relative_month', 'relative_week']
print("Null counts:")
print(df_wondr[critical_cols].isnull().sum())

Null counts:
review_text_cleaned    0
relative_month         0
relative_week          0
dtype: int64


In [6]:
# Sanity check: relative_month coverage (should be 1-12)
print("Relative month distribution:")
print(df_wondr['relative_month'].value_counts().sort_index())

Relative month distribution:
relative_month
1      302
2      395
3      544
4      678
5     2568
6     1297
7      727
8      621
9      462
10     357
11     388
12     643
Name: count, dtype: int64


### 1.2 Extract texts as list

BERTopic expects a list of strings, not a pandas Series. We extract once here and reuse throughout the notebook.

In [7]:
docs_wondr = df_wondr['review_text_cleaned'].tolist()

print(f"Total docs: {len(docs_wondr)}")
print(f"Sample doc: {docs_wondr[0][:200]}")

Total docs: 8982
Sample doc: kok tidak bisa di screenshot bagaimana caranya kasih bukti ke penerima bagaimana sih ini aplikasi


### 1.3 Load IndoBERT model

Loaded once and reused for all embedding operations. First call may take a few seconds (loading from local HuggingFace cache).

In [9]:
model = load_indobert_model()
print(f"Model loaded: {type(model).__name__}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded: SentenceTransformer


### 1.4 Generate embeddings (or load from cache)

**First run:** generate from scratch, save to `data/embeddings/wondr_embeddings.npy` (~20-30 min on CPU).

**Subsequent runs:** load from cache (instant).

If preprocessing data changes, manually delete the cache file before re-running this cell.

In [10]:
embedding_cache_path = Path('data/embeddings/wondr_embeddings.npy')
embedding_cache_path.parent.mkdir(parents=True, exist_ok=True)

embeddings_wondr = generate_embeddings(
    texts=docs_wondr,
    model=model,
    cache_path=embedding_cache_path,
)

print(f"\nEmbedding shape: {embeddings_wondr.shape}")
print(f"Dtype: {embeddings_wondr.dtype}")
print(f"Cache file size: {embedding_cache_path.stat().st_size / (1024**2):.1f} MB")

[cache miss] Generating embeddings for 8982 texts...


Batches:   0%|          | 0/281 [00:00<?, ?it/s]

[cache miss] Saved to data\embeddings\wondr_embeddings.npy, shape: (8982, 768)

Embedding shape: (8982, 768)
Dtype: float32
Cache file size: 26.3 MB
